In [23]:
import pandas as pd
import numpy as np

In [24]:
df = pd.read_csv('./clean_data_after_eda.csv')
df["date_activ"] = pd.to_datetime(df["date_activ"], format='%Y-%m-%d')
df["date_end"] = pd.to_datetime(df["date_end"], format='%Y-%m-%d')
df["date_modif_prod"] = pd.to_datetime(df["date_modif_prod"], format='%Y-%m-%d')
df["date_renewal"] = pd.to_datetime(df["date_renewal"], format='%Y-%m-%d')

In [3]:
df.head(3)

,id,channel_sales,cons_12m,cons_gas_12m,cons_last_month,date_activ,date_end,date_modif_prod,date_renewal,forecast_cons_12m,...,var_6m_price_off_peak_var,var_6m_price_peak_var,var_6m_price_mid_peak_var,var_6m_price_off_peak_fix,var_6m_price_peak_fix,var_6m_price_mid_peak_fix,var_6m_price_off_peak,var_6m_price_peak,var_6m_price_mid_peak,churn
0,24011ae4ebbe3035111d65fa7c15bc57,foosdfpfkusacimwkcsosbicdxkicaua,0,54946,0,2013-06-15,2016-06-15,2015-11-01,2015-06-23,0.00,...,0.000131,4.100838e-05,0.000908,2.086294,99.530517,44.235794,2.086425,9.953056e+01,44.236702,1
1,d29c2c54acc38ff3c0614d0a653813dd,MISSING,4660,0,0,2009-08-21,2016-08-30,2009-08-21,2015-08-31,189.95,...,0.000003,1.217891e-03,0.000000,0.009482,0.000000,0.000000,0.009485,1.217891e-03,0.000000,0
2,764c75f661154dac3a6c254cd082ea7d,foosdfpfkusacimwkcsosbicdxkicaua,544,0,0,2010-04-16,2016-04-16,2010-04-16,2015-04-17,47.96,...,0.000004,9.450150e-08,0.000000,0.000000,0.000000,0.000000,0.000004,9.450150e-08,0.000000,0


In [25]:
price_df = pd.read_csv('price_data.csv')
price_df["price_date"] = pd.to_datetime(price_df["price_date"], format='%Y-%m-%d')
price_df.head()

,id,price_date,price_off_peak_var,price_peak_var,price_mid_peak_var,price_off_peak_fix,price_peak_fix,price_mid_peak_fix
0,038af19179925da21a25619c5a24b745,2015-01-01,0.151367,0.0,0.0,44.266931,0.0,0.0
1,038af19179925da21a25619c5a24b745,2015-02-01,0.151367,0.0,0.0,44.266931,0.0,0.0
2,038af19179925da21a25619c5a24b745,2015-03-01,0.151367,0.0,0.0,44.266931,0.0,0.0
3,038af19179925da21a25619c5a24b745,2015-04-01,0.149626,0.0,0.0,44.266931,0.0,0.0
4,038af19179925da21a25619c5a24b745,2015-05-01,0.149626,0.0,0.0,44.266931,0.0,0.0


In [26]:
# Group off-peak prices by companies and month
monthly_price_by_id = price_df.groupby(['id', 'price_date']).agg({'price_off_peak_var': 'mean', 'price_off_peak_fix': 'mean'}).reset_index()

# Get january and december prices
jan_prices = monthly_price_by_id.groupby('id').first().reset_index()
dec_prices = monthly_price_by_id.groupby('id').last().reset_index()

# Calculate the difference
diff = pd.merge(dec_prices.rename(columns={'price_off_peak_var': 'dec_1', 'price_off_peak_fix': 'dec_2'}), jan_prices.drop(columns='price_date'), on='id')
diff['offpeak_diff_dec_january_energy'] = diff['dec_1'] - diff['price_off_peak_var']
diff['offpeak_diff_dec_january_power'] = diff['dec_2'] - diff['price_off_peak_fix']
diff = diff[['id', 'offpeak_diff_dec_january_energy','offpeak_diff_dec_january_power']]
diff.head()

,id,offpeak_diff_dec_january_energy,offpeak_diff_dec_january_power
0,0002203ffbb812588b632b9e628cc38d,-0.006192,0.162916
1,0004351ebdd665e6ee664792efc4fd13,-0.004104,0.177779
2,0010bcc39e42b3c2131ed2ce55246e3c,0.050443,1.500000
3,0010ee3855fdea87602a5b7aba8e42de,-0.010018,0.162916
4,00114d74e963e47177db89bc70108537,-0.003994,-0.000001


In [27]:
# Merge engineered price difference features

df = pd.merge(
    df,
    diff,
    on='id',
    how='left'
)

df.head()

,id,channel_sales,cons_12m,cons_gas_12m,cons_last_month,date_activ,date_end,date_modif_prod,date_renewal,forecast_cons_12m,...,var_6m_price_mid_peak_var,var_6m_price_off_peak_fix,var_6m_price_peak_fix,var_6m_price_mid_peak_fix,var_6m_price_off_peak,var_6m_price_peak,var_6m_price_mid_peak,churn,offpeak_diff_dec_january_energy,offpeak_diff_dec_january_power
0,24011ae4ebbe3035111d65fa7c15bc57,foosdfpfkusacimwkcsosbicdxkicaua,0,54946,0,2013-06-15,2016-06-15,2015-11-01,2015-06-23,0.00,...,9.084737e-04,2.086294,99.530517,44.235794,2.086425,9.953056e+01,4.423670e+01,1,0.020057,3.700961
1,d29c2c54acc38ff3c0614d0a653813dd,MISSING,4660,0,0,2009-08-21,2016-08-30,2009-08-21,2015-08-31,189.95,...,0.000000e+00,0.009482,0.000000,0.000000,0.009485,1.217891e-03,0.000000e+00,0,-0.003767,0.177779
2,764c75f661154dac3a6c254cd082ea7d,foosdfpfkusacimwkcsosbicdxkicaua,544,0,0,2010-04-16,2016-04-16,2010-04-16,2015-04-17,47.96,...,0.000000e+00,0.000000,0.000000,0.000000,0.000004,9.450150e-08,0.000000e+00,0,-0.004670,0.177779
3,bba03439a292a1e166f80264c16191cb,lmkebamcaaclubfxadlmueccxoimlema,1584,0,0,2010-03-30,2016-03-30,2010-03-30,2015-03-31,240.04,...,0.000000e+00,0.000000,0.000000,0.000000,0.000003,0.000000e+00,0.000000e+00,0,-0.004547,0.177779
4,149d57cf92fc41cf94415803a877cb4b,MISSING,4425,0,526,2010-01-13,2016-03-07,2010-01-13,2015-03-09,445.75,...,4.860000e-10,0.000000,0.000000,0.000000,0.000011,2.896760e-06,4.860000e-10,0,-0.006192,0.162916


In [28]:
df['total_consumption'] = (
    df['cons_12m']
    +
    df['cons_gas_12m']
)

In [29]:
df['avg_monthly_consumption'] = (
    df['cons_12m'] / 12
)

In [30]:
df['consumption_ratio'] = (
    df['cons_last_month']
    /
    (df['cons_12m'] + 1)
)

In [31]:
df['profitability'] = (
    df['net_margin']
    /
    (df['cons_12m'] + 1)
)

In [32]:
df['remaining_contract_days'] = (
    df['date_end']
    -
    df['date_renewal']
).dt.days

In [33]:
df['loyalty_years'] = (
    df['num_years_antig']
)

In [34]:
df['price_sensitivity'] = (
    df['offpeak_diff_dec_january_energy']
    *
    df['cons_12m']
)

In [35]:
df['discount_impact'] = (
    df['forecast_discount_energy']
    *
    df['forecast_cons_12m']
)

In [36]:
df['revenue_per_power'] = (
    df['net_margin']
    /
    (df['pow_max'] + 1)
)

In [37]:
df['products_per_year'] = (
    df['nb_prod_act']
    /
    (df['num_years_antig'] + 1)
)

In [38]:
df['high_consumption'] = np.where(
    df['cons_12m'] > df['cons_12m'].median(),
    1,
    0
)

In [39]:
df['high_net_margin'] = np.where(
    df['net_margin'] > df['net_margin'].median(),
    1,
    0
)

In [40]:
df['contract_length'] = (
    df['date_end']
    -
    df['date_activ']
).dt.days

In [41]:
df['days_since_modification'] = (
    df['date_renewal']
    -
    df['date_modif_prod']
).dt.days

In [42]:
df.head()

,id,channel_sales,cons_12m,cons_gas_12m,cons_last_month,date_activ,date_end,date_modif_prod,date_renewal,forecast_cons_12m,...,remaining_contract_days,loyalty_years,price_sensitivity,discount_impact,revenue_per_power,products_per_year,high_consumption,high_net_margin,contract_length,days_since_modification
0,24011ae4ebbe3035111d65fa7c15bc57,foosdfpfkusacimwkcsosbicdxkicaua,0,54946,0,2013-06-15,2016-06-15,2015-11-01,2015-06-23,0.00,...,358,3,0.000000,0.0,15.207624,0.500000,0,1,1096,-131
1,d29c2c54acc38ff3c0614d0a653813dd,MISSING,4660,0,0,2009-08-21,2016-08-30,2009-08-21,2015-08-31,189.95,...,365,6,-17.554220,0.0,1.276351,0.142857,0,0,2566,2201
2,764c75f661154dac3a6c254cd082ea7d,foosdfpfkusacimwkcsosbicdxkicaua,544,0,0,2010-04-16,2016-04-16,2010-04-16,2015-04-17,47.96,...,365,6,-2.540480,0.0,0.444265,0.142857,0,0,2192,1827
3,bba03439a292a1e166f80264c16191cb,lmkebamcaaclubfxadlmueccxoimlema,1584,0,0,2010-03-30,2016-03-30,2010-03-30,2015-03-31,240.04,...,365,6,-7.202448,0.0,1.792958,0.142857,0,0,2192,1827
4,149d57cf92fc41cf94415803a877cb4b,MISSING,4425,0,526,2010-01-13,2016-03-07,2010-01-13,2015-03-09,445.75,...,364,6,-27.399600,0.0,2.306731,0.142857,0,0,2245,1881


In [44]:
new_features = [
    'offpeak_diff_dec_january_energy',
    'offpeak_diff_dec_january_power',
    'total_consumption',
    'avg_monthly_consumption',
    'consumption_ratio',
    'profitability',
    'remaining_contract_days',
    'loyalty_years',
    'price_sensitivity',
    'discount_impact',
    'revenue_per_power',
    'products_per_year',
    'high_consumption',
    'high_net_margin',
    'contract_length',
    'days_since_modification'
]

df[new_features].head()

,offpeak_diff_dec_january_energy,offpeak_diff_dec_january_power,total_consumption,avg_monthly_consumption,consumption_ratio,profitability,remaining_contract_days,loyalty_years,price_sensitivity,discount_impact,revenue_per_power,products_per_year,high_consumption,high_net_margin,contract_length,days_since_modification
0,0.020057,3.700961,54946,0.000000,0.000000,678.990000,358,3,0.000000,0.0,15.207624,0.500000,0,1,1096,-131
1,-0.003767,0.177779,4660,388.333333,0.000000,0.004053,365,6,-17.554220,0.0,1.276351,0.142857,0,0,2566,2201
2,-0.004670,0.177779,544,45.333333,0.000000,0.012110,365,6,-2.540480,0.0,0.444265,0.142857,0,0,2192,1827
3,-0.004547,0.177779,1584,132.000000,0.000000,0.016063,365,6,-7.202448,0.0,1.792958,0.142857,0,0,2192,1827
4,-0.006192,0.162916,4425,368.750000,0.118843,0.010840,364,6,-27.399600,0.0,2.306731,0.142857,0,0,2245,1881
